<a href="https://colab.research.google.com/github/evakonstantinova/EfficientNet-B0/blob/main/EfficientNet_B0_QuantumNoise_FreeSimulation_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# DATASET DOWNLOAD AND STRATIFIED SPLIT

from pathlib import Path
from collections import Counter

import kagglehub
from sklearn.model_selection import train_test_split

path = kagglehub.dataset_download(
    "masoudnickparvar/brain-tumor-mri-dataset"
)

classes = ["glioma", "meningioma", "notumor", "pituitary"]

all_files = []
all_labels = []

for class_name in classes:
    for folder in ["Training", "Testing"]:
        class_path = Path(path) / folder / class_name

        for file_path in class_path.iterdir():
            if file_path.is_file():
                all_files.append(str(file_path))
                all_labels.append(class_name)

train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files,
    all_labels,
    test_size=0.30,
    random_state=42,
    stratify=all_labels
)

val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files,
    temp_labels,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

print("Dataset path:", path)
print("Total images:", len(all_files))
print("Overall distribution:", Counter(all_labels))
print("Training:", len(train_files), Counter(train_labels))
print("Validation:", len(val_files), Counter(val_labels))
print("Testing:", len(test_files), Counter(test_labels))

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Dataset path: /kaggle/input/brain-tumor-mri-dataset
Total images: 7200
Overall distribution: Counter({'glioma': 1800, 'meningioma': 1800, 'notumor': 1800, 'pituitary': 1800})
Training: 5040 Counter({'notumor': 1260, 'pituitary': 1260, 'glioma': 1260, 'meningioma': 1260})
Validation: 1080 Counter({'notumor': 270, 'glioma': 270, 'meningioma': 270, 'pituitary': 270})
Testing: 1080 Counter({'meningioma': 270, 'notumor': 270, 'glioma': 270, 'pituitary': 270})


In [2]:
# DATA PREPROCESSING AND DATALOADERS

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class_to_idx = {
    "glioma": 0,
    "meningioma": 1,
    "notumor": 2,
    "pituitary": 3
}

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class BrainTumorDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image = Image.open(self.files[idx]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_to_idx[self.labels[idx]]
        return image, label

train_dataset = BrainTumorDataset(
    train_files,
    train_labels,
    transform=train_transform
)

val_dataset = BrainTumorDataset(
    val_files,
    val_labels,
    transform=val_test_transform
)

test_dataset = BrainTumorDataset(
    test_files,
    test_labels,
    transform=val_test_transform
)

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 5040
Validation dataset: 1080
Test dataset: 1080


In [3]:
# PENNYLANE DOWNLOAD
!pip -q install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 93.4 MB/s eta 0:00:00


In [4]:
# IMPORTING LIBRARIES AND CHECKING VERSIONS

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision.models import EfficientNet_B0_Weights

import pennylane as qml

print("PyTorch version:", torch.__version__)
print("PennyLane version:", qml.__version__)

PyTorch version: 2.11.0+cu128
PennyLane version: 0.45.1


In [5]:
# DEFINING THE QUANTUM CIRCUIT

# DEFINE THE QUANTUM CIRCUIT

N_QUBITS = 4
N_Q_LAYERS = 2

quantum_device = qml.device(
    "default.qubit",
    wires=N_QUBITS
)

@qml.qnode(
    quantum_device,
    interface="torch",
    diff_method="backprop"
)
def quantum_circuit(inputs, weights):

    # ENCODE 4 CLASSICAL FEATURES INTO 4 QUBITS
    qml.AngleEmbedding(
        inputs,
        wires=range(N_QUBITS),
        rotation="Y"
    )

    # APPLY TRAINABLE QUANTUM LAYERS
    qml.StronglyEntanglingLayers(
        weights,
        wires=range(N_QUBITS)
    )

    # MEASURE EACH QUBIT
    return [
        qml.expval(qml.PauliZ(i))
        for i in range(N_QUBITS)
    ]


weight_shapes = {
    "weights": (
        N_Q_LAYERS,
        N_QUBITS,
        3
    )
}

quantum_layer = qml.qnn.TorchLayer(
    quantum_circuit,
    weight_shapes
)

print("Quantum layer created successfully.")
print("Qubits:", N_QUBITS)
print("Quantum layers:", N_Q_LAYERS)

Quantum layer created successfully.
Qubits: 4
Quantum layers: 2


In [6]:
# BUILD THE EFFICIENTNET-B0 + QUANTUM HYBRID MODEL

# BUILD THE EFFICIENTNET-B0 + QUANTUM HYBRID MODEL

class HQNN(nn.Module):

    def __init__(self, quantum_layer):
        super().__init__()

        # LOAD IMAGENET-PRETRAINED EFFICIENTNET-B0
        efficientnet = models.efficientnet_b0(
            weights=EfficientNet_B0_Weights.DEFAULT
        )

        # KEEP ONLY THE FEATURE EXTRACTOR
        self.features = efficientnet.features
        self.avgpool = efficientnet.avgpool

        # FREEZE EFFICIENTNET FEATURE EXTRACTOR
        for param in self.features.parameters():
            param.requires_grad = False

        # REDUCE 1280 EFFICIENTNET FEATURES TO 4
        self.feature_reduction = nn.Linear(
            1280,
            N_QUBITS
        )

        # QUANTUM LAYER
        self.quantum_layer = quantum_layer

        # FINAL FOUR-CLASS CLASSIFIER
        self.classifier = nn.Linear(
            N_QUBITS,
            4
        )

    def forward(self, x):

        # EXTRACT EFFICIENTNET FEATURES
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        # REDUCE 1280 FEATURES TO 4
        x = self.feature_reduction(x)

        # SCALE VALUES FOR QUANTUM ANGLE ENCODING
        x = torch.tanh(x) * torch.pi

        # MOVE QUANTUM INPUT TO CPU
        x = x.cpu()

        # RUN THE QUANTUM CIRCUIT ON CPU
        x = self.quantum_layer(x)

        # MOVE QUANTUM OUTPUT BACK TO THE CLASSIFIER DEVICE
        classifier_device = next(
            self.classifier.parameters()
        ).device

        x = x.to(classifier_device)

        # PRODUCE FOUR-CLASS OUTPUT
        x = self.classifier(x)

        return x


hqnn_model = HQNN(
    quantum_layer
)

print(hqnn_model)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 79.2MB/s]


HQNN(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivation(
   

In [7]:
# COUNT TOTAL AND TRAINABLE PARAMETERS

total_params = sum(
    p.numel()
    for p in hqnn_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in hqnn_model.parameters()
    if p.requires_grad
)

quantum_params = sum(
    p.numel()
    for p in hqnn_model.quantum_layer.parameters()
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Quantum trainable parameters: {quantum_params:,}")

Total parameters: 4,012,716
Trainable parameters: 5,168
Quantum trainable parameters: 24


In [8]:
# TEST THE HQNN WITH ONE SMALL BATCH

hqnn_model.eval()

images, labels = next(iter(train_loader))

# USE ONLY 2 IMAGES FOR THE TEST
test_images = images[:2]

with torch.no_grad():
    outputs = hqnn_model(test_images)

print("Input shape:", test_images.shape)
print("Output shape:", outputs.shape)
print("Output:")
print(outputs)

Input shape: torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 4])
Output:
tensor([[-0.5933,  0.5411, -0.3995,  0.0604],
        [-0.6283,  0.5490, -0.3288,  0.0026]])


In [9]:
# PREPARE THE FROZEN EFFICIENTNET-B0 FEATURE EXTRACTOR

feature_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Feature extraction device:", feature_device)

# MOVE ONLY THE FROZEN EFFICIENTNET PART TO THE AVAILABLE DEVICE
hqnn_model.features = hqnn_model.features.to(feature_device)
hqnn_model.avgpool = hqnn_model.avgpool.to(feature_device)

# KEEP THE FROZEN FEATURE EXTRACTOR IN EVALUATION MODE
hqnn_model.features.eval()
hqnn_model.avgpool.eval()


def extract_efficientnet_features(data_loader):

    all_features = []
    all_labels = []

    # NO GRADIENTS ARE REQUIRED FOR THE FROZEN EFFICIENTNET FEATURE EXTRACTOR
    with torch.no_grad():

        for images, labels in data_loader:

            images = images.to(feature_device)

            # EXTRACT EFFICIENTNET-B0 FEATURES
            features = hqnn_model.features(images)
            features = hqnn_model.avgpool(features)
            features = torch.flatten(features, 1)

            # MOVE THE 1280-DIMENSIONAL FEATURES BACK TO CPU
            all_features.append(features.cpu())
            all_labels.append(labels.cpu())

    all_features = torch.cat(all_features, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return all_features, all_labels


print("Feature extraction function created successfully.")

Feature extraction device: cuda
Feature extraction function created successfully.


In [10]:
# CONFIGURE THE NOISE-FREE HQNN TRAINING

# KEEP THE TRAINABLE HYBRID CLASSIFIER ON CPU FOR THE PENNYLANE SIMULATOR
hqnn_model.feature_reduction = hqnn_model.feature_reduction.cpu()
hqnn_model.quantum_layer = hqnn_model.quantum_layer.cpu()
hqnn_model.classifier = hqnn_model.classifier.cpu()

# DEFINE THE LOSS FUNCTION
criterion = nn.CrossEntropyLoss()

# OPTIMIZE ONLY THE TRAINABLE HQNN PARAMETERS
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, hqnn_model.parameters()),
    lr=0.001
)

# SET THE NUMBER OF TRAINING EPOCHS
NUM_EPOCHS = 10

# INITIALIZE BEST VALIDATION LOSS
best_val_loss = float("inf")

print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Number of epochs:", NUM_EPOCHS)
print("Noise-free quantum device: default.qubit")

Loss function: CrossEntropyLoss
Optimizer: Adam
Learning rate: 0.001
Number of epochs: 10
Noise-free quantum device: default.qubit


In [11]:
# EXTRACT EFFICIENTNET-B0 FEATURES FROM THE VALIDATION DATASET

print("Extracting validation features...")

val_features, val_labels = extract_efficientnet_features(val_loader)

print("Validation feature extraction completed.")
print("Validation features shape:", val_features.shape)
print("Validation labels shape:", val_labels.shape)

Extracting validation features...
Validation feature extraction completed.
Validation features shape: torch.Size([1080, 1280])
Validation labels shape: torch.Size([1080])


In [12]:
# TRAIN THE NOISE-FREE HQNN

import time
from sklearn.metrics import f1_score
from torch.utils.data import TensorDataset, DataLoader

# STORE TRAINING HISTORY
train_losses = []
train_accuracies = []
train_f1_scores = []

val_losses = []
val_accuracies = []
val_f1_scores = []

best_val_loss = float("inf")
best_epoch = 0

# START TOTAL TRAINING TIMER
training_start_time = time.time()

for epoch in range(NUM_EPOCHS):

    print(f"\nEPOCH {epoch + 1}/{NUM_EPOCHS}")
    print("Extracting augmented training features...")

    # EXTRACT NEW TRAINING FEATURES EACH EPOCH
    # THIS PRESERVES RANDOM TRAINING AUGMENTATION
    train_features, train_labels = extract_efficientnet_features(
        train_loader
    )

    # CREATE A FEATURE-LEVEL TRAINING LOADER
    train_feature_dataset = TensorDataset(
        train_features,
        train_labels
    )

    train_feature_loader = DataLoader(
        train_feature_dataset,
        batch_size=32,
        shuffle=True
    )

    # SET THE TRAINABLE HQNN COMPONENTS TO TRAINING MODE
    hqnn_model.feature_reduction.train()
    hqnn_model.quantum_layer.train()
    hqnn_model.classifier.train()

    running_train_loss = 0.0
    train_predictions = []
    train_targets = []

    # TRAIN THE HYBRID CLASSIFIER
    for features, labels in train_feature_loader:

        optimizer.zero_grad()

        # REDUCE 1280 EFFICIENTNET FEATURES TO 4
        outputs = hqnn_model.feature_reduction(features)

        # SCALE THE FOUR FEATURES FOR QUANTUM ANGLE ENCODING
        outputs = torch.tanh(outputs) * torch.pi

        # PASS THE FEATURES THROUGH THE QUANTUM CIRCUIT
        outputs = hqnn_model.quantum_layer(outputs)

        # PRODUCE FOUR-CLASS OUTPUT
        outputs = hqnn_model.classifier(outputs)

        # CALCULATE CLASSIFICATION LOSS
        loss = criterion(outputs, labels)

        # CALCULATE GRADIENTS AND UPDATE TRAINABLE PARAMETERS
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * features.size(0)

        predictions = torch.argmax(outputs, dim=1)

        train_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        train_targets.extend(
            labels.cpu().numpy()
        )

    # CALCULATE TRAINING METRICS
    epoch_train_loss = (
        running_train_loss / len(train_feature_dataset)
    )

    epoch_train_accuracy = (
        sum(
            p == t
            for p, t in zip(
                train_predictions,
                train_targets
            )
        )
        / len(train_targets)
    )

    epoch_train_f1 = f1_score(
        train_targets,
        train_predictions,
        average="macro"
    )

    # SET THE TRAINABLE HQNN COMPONENTS TO EVALUATION MODE
    hqnn_model.feature_reduction.eval()
    hqnn_model.quantum_layer.eval()
    hqnn_model.classifier.eval()

    # CREATE THE VALIDATION FEATURE LOADER
    val_feature_dataset = TensorDataset(
        val_features,
        val_labels
    )

    val_feature_loader = DataLoader(
        val_feature_dataset,
        batch_size=32,
        shuffle=False
    )

    running_val_loss = 0.0
    val_predictions = []
    val_targets = []

    # EVALUATE ON THE VALIDATION DATASET
    with torch.no_grad():

        for features, labels in val_feature_loader:

            # REDUCE 1280 FEATURES TO 4
            outputs = hqnn_model.feature_reduction(features)

            # SCALE FEATURES FOR QUANTUM ANGLE ENCODING
            outputs = torch.tanh(outputs) * torch.pi

            # PASS THROUGH THE NOISE-FREE QUANTUM CIRCUIT
            outputs = hqnn_model.quantum_layer(outputs)

            # PRODUCE FOUR-CLASS OUTPUT
            outputs = hqnn_model.classifier(outputs)

            loss = criterion(outputs, labels)

            running_val_loss += (
                loss.item() * features.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_predictions.extend(
                predictions.cpu().numpy()
            )

            val_targets.extend(
                labels.cpu().numpy()
            )

    # CALCULATE VALIDATION METRICS
    epoch_val_loss = (
        running_val_loss / len(val_feature_dataset)
    )

    epoch_val_accuracy = (
        sum(
            p == t
            for p, t in zip(
                val_predictions,
                val_targets
            )
        )
        / len(val_targets)
    )

    epoch_val_f1 = f1_score(
        val_targets,
        val_predictions,
        average="macro"
    )

    # STORE EPOCH RESULTS
    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_accuracy)
    train_f1_scores.append(epoch_train_f1)

    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_accuracy)
    val_f1_scores.append(epoch_val_f1)

    # SAVE THE CHECKPOINT WITH THE LOWEST VALIDATION LOSS
    if epoch_val_loss < best_val_loss:

        best_val_loss = epoch_val_loss
        best_epoch = epoch + 1

        torch.save(
            hqnn_model.state_dict(),
            "/content/best_hqnn_noisefree.pth"
        )

        checkpoint_message = " <-- BEST CHECKPOINT"

    else:
        checkpoint_message = ""

    # PRINT EPOCH RESULTS
    print(
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Train Acc: {epoch_train_accuracy:.4f} | "
        f"Train Macro F1: {epoch_train_f1:.4f}"
    )

    print(
        f"Val Loss:   {epoch_val_loss:.4f} | "
        f"Val Acc:   {epoch_val_accuracy:.4f} | "
        f"Val Macro F1:   {epoch_val_f1:.4f}"
        f"{checkpoint_message}"
    )


# CALCULATE TOTAL TRAINING WALL TIME
training_end_time = time.time()

hqnn_training_time = (
    training_end_time - training_start_time
)

minutes = int(hqnn_training_time // 60)
seconds = int(hqnn_training_time % 60)

print("\nNOISE-FREE HQNN TRAINING COMPLETED")
print("Best checkpoint epoch:", best_epoch)
print(f"Best validation loss: {best_val_loss:.4f}")
print(
    f"Total training wall time: "
    f"{minutes} min {seconds} s"
)


EPOCH 1/10
Extracting augmented training features...
Train Loss: 1.2970 | Train Acc: 0.3798 | Train Macro F1: 0.2572
Val Loss:   1.1972 | Val Acc:   0.3963 | Val Macro F1:   0.2619 <-- BEST CHECKPOINT

EPOCH 2/10
Extracting augmented training features...
Train Loss: 1.0953 | Train Acc: 0.6091 | Train Macro F1: 0.5979
Val Loss:   0.9790 | Val Acc:   0.8074 | Val Macro F1:   0.8008 <-- BEST CHECKPOINT

EPOCH 3/10
Extracting augmented training features...
Train Loss: 0.8567 | Train Acc: 0.8317 | Train Macro F1: 0.8248
Val Loss:   0.7586 | Val Acc:   0.8315 | Val Macro F1:   0.8307 <-- BEST CHECKPOINT

EPOCH 4/10
Extracting augmented training features...
Train Loss: 0.6308 | Train Acc: 0.8528 | Train Macro F1: 0.8501
Val Loss:   0.5647 | Val Acc:   0.8491 | Val Macro F1:   0.8476 <-- BEST CHECKPOINT

EPOCH 5/10
Extracting augmented training features...
Train Loss: 0.4633 | Train Acc: 0.8813 | Train Macro F1: 0.8799
Val Loss:   0.5005 | Val Acc:   0.8454 | Val Macro F1:   0.8397 <-- BEST C

In [13]:
# LOAD THE BEST STAGE 1 HQNN CHECKPOINT

hqnn_model.load_state_dict(
    torch.load(
        "/content/best_hqnn_noisefree.pth",
        weights_only=True
    )
)

print("Best Stage 1 HQNN checkpoint loaded.")


# FREEZE THE COMPLETE EFFICIENTNET-B0 FEATURE EXTRACTOR FIRST

for param in hqnn_model.features.parameters():
    param.requires_grad = False


# UNFREEZE THE FINAL TWO EFFICIENTNET-B0 FEATURE SECTIONS

for param in hqnn_model.features[-2:].parameters():
    param.requires_grad = True


# KEEP THE FEATURE REDUCTION LAYER TRAINABLE

for param in hqnn_model.feature_reduction.parameters():
    param.requires_grad = True


# KEEP THE QUANTUM CIRCUIT TRAINABLE

for param in hqnn_model.quantum_layer.parameters():
    param.requires_grad = True


# KEEP THE FINAL CLASSIFIER TRAINABLE

for param in hqnn_model.classifier.parameters():
    param.requires_grad = True


# COUNT TOTAL AND TRAINABLE PARAMETERS FOR FINE-TUNING

total_params_finetune = sum(
    p.numel()
    for p in hqnn_model.parameters()
)

trainable_params_finetune = sum(
    p.numel()
    for p in hqnn_model.parameters()
    if p.requires_grad
)

quantum_params_finetune = sum(
    p.numel()
    for p in hqnn_model.quantum_layer.parameters()
    if p.requires_grad
)


print(f"Total parameters: {total_params_finetune:,}")
print(
    f"Fine-tuning trainable parameters: "
    f"{trainable_params_finetune:,}"
)
print(
    f"Quantum trainable parameters: "
    f"{quantum_params_finetune:,}"
)

Best Stage 1 HQNN checkpoint loaded.
Total parameters: 4,012,716
Fine-tuning trainable parameters: 1,134,560
Quantum trainable parameters: 24


In [14]:
# CONFIGURE HQNN FINE-TUNING

# SELECT CUDA FOR THE CLASSICAL COMPONENTS
finetune_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# MOVE THE COMPLETE MODEL TO THE SELECTED DEVICE FIRST
hqnn_model = hqnn_model.to(
    finetune_device
)

# MOVE THE QUANTUM LAYER BACK TO CPU
hqnn_model.quantum_layer = (
    hqnn_model.quantum_layer.cpu()
)

# DEFINE THE LOSS FUNCTION
criterion = nn.CrossEntropyLoss()

# SEPARATE CLASSICAL AND QUANTUM TRAINABLE PARAMETERS
classical_trainable_params = [
    param
    for name, param in hqnn_model.named_parameters()
    if param.requires_grad
    and not name.startswith("quantum_layer")
]

quantum_trainable_params = list(
    hqnn_model.quantum_layer.parameters()
)

# CONFIGURE ADAMW FOR MIXED CPU AND GPU PARAMETERS
optimizer = torch.optim.AdamW(
    [
        {
            "params": classical_trainable_params
        },
        {
            "params": quantum_trainable_params
        }
    ],
    lr=0.0001,
    weight_decay=0.0001,
    foreach=False
)

# SET THE MAXIMUM NUMBER OF FINE-TUNING EPOCHS
FINE_TUNE_EPOCHS = 10

# CONFIGURE EARLY STOPPING
PATIENCE = 3
MIN_DELTA = 0.001
EARLY_STOPPING_START_EPOCH = 6

best_finetune_val_loss = float("inf")
best_finetune_epoch = 0
patience_counter = 0

# COUNT TRAINABLE PARAMETERS
trainable_params_finetune = sum(
    p.numel()
    for p in hqnn_model.parameters()
    if p.requires_grad
)

print(
    "Classical fine-tuning device:",
    finetune_device
)

print(
    "Quantum layer device:",
    next(
        hqnn_model.quantum_layer.parameters()
    ).device
)

print("Optimizer: AdamW")
print("Learning rate: 0.0001")
print("Weight decay: 0.0001")
print("Maximum epochs:", FINE_TUNE_EPOCHS)
print("Early stopping patience:", PATIENCE)
print("Early stopping min_delta:", MIN_DELTA)

print(
    "Fine-tuning trainable parameters:",
    f"{trainable_params_finetune:,}"
)

Classical fine-tuning device: cuda
Quantum layer device: cpu
Optimizer: AdamW
Learning rate: 0.0001
Weight decay: 0.0001
Maximum epochs: 10
Early stopping patience: 3
Early stopping min_delta: 0.001
Fine-tuning trainable parameters: 1,134,560


In [15]:
# FINE-TUNE THE NOISE-FREE HQNN END-TO-END

import time
from sklearn.metrics import f1_score

# STORE FINE-TUNING HISTORY
finetune_train_losses = []
finetune_train_accuracies = []
finetune_train_f1_scores = []

finetune_val_losses = []
finetune_val_accuracies = []
finetune_val_f1_scores = []

best_finetune_val_loss = float("inf")
best_finetune_epoch = 0

patience_counter = 0
early_stopping_best_loss = float("inf")

# START THE FINE-TUNING TIMER
finetune_start_time = time.time()

for epoch in range(FINE_TUNE_EPOCHS):

    # SET THE COMPLETE HQNN TO TRAINING MODE
    hqnn_model.train()

    running_train_loss = 0.0
    train_predictions = []
    train_targets = []

    # TRAIN THE HQNN ON THE MRI TRAINING DATASET
    for images, labels in train_loader:

        # MOVE THE BATCH TO THE FINE-TUNING DEVICE
        images = images.to(finetune_device)
        labels = labels.to(finetune_device)

        # RESET GRADIENTS
        optimizer.zero_grad()

        # RUN THE COMPLETE HYBRID MODEL
        outputs = hqnn_model(images)

        # CALCULATE CLASSIFICATION LOSS
        loss = criterion(
            outputs,
            labels
        )

        # CALCULATE GRADIENTS
        loss.backward()

        # UPDATE THE TRAINABLE PARAMETERS
        optimizer.step()

        running_train_loss += (
            loss.item() * images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        train_targets.extend(
            labels.detach().cpu().numpy()
        )

    # CALCULATE TRAINING METRICS
    epoch_train_loss = (
        running_train_loss
        / len(train_loader.dataset)
    )

    epoch_train_accuracy = (
        sum(
            p == t
            for p, t in zip(
                train_predictions,
                train_targets
            )
        )
        / len(train_targets)
    )

    epoch_train_f1 = f1_score(
        train_targets,
        train_predictions,
        average="macro"
    )

    # SWITCH THE HQNN TO EVALUATION MODE
    hqnn_model.eval()

    running_val_loss = 0.0
    val_predictions = []
    val_targets = []

    # EVALUATE THE HQNN ON THE VALIDATION DATASET
    with torch.no_grad():

        for images, labels in val_loader:

            # MOVE THE BATCH TO THE FINE-TUNING DEVICE
            images = images.to(finetune_device)
            labels = labels.to(finetune_device)

            # RUN THE COMPLETE HYBRID MODEL
            outputs = hqnn_model(images)

            # CALCULATE VALIDATION LOSS
            loss = criterion(
                outputs,
                labels
            )

            running_val_loss += (
                loss.item() * images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_predictions.extend(
                predictions.detach().cpu().numpy()
            )

            val_targets.extend(
                labels.detach().cpu().numpy()
            )

    # CALCULATE VALIDATION METRICS
    epoch_val_loss = (
        running_val_loss
        / len(val_loader.dataset)
    )

    epoch_val_accuracy = (
        sum(
            p == t
            for p, t in zip(
                val_predictions,
                val_targets
            )
        )
        / len(val_targets)
    )

    epoch_val_f1 = f1_score(
        val_targets,
        val_predictions,
        average="macro"
    )

    # STORE THE EPOCH RESULTS
    finetune_train_losses.append(
        epoch_train_loss
    )

    finetune_train_accuracies.append(
        epoch_train_accuracy
    )

    finetune_train_f1_scores.append(
        epoch_train_f1
    )

    finetune_val_losses.append(
        epoch_val_loss
    )

    finetune_val_accuracies.append(
        epoch_val_accuracy
    )

    finetune_val_f1_scores.append(
        epoch_val_f1
    )

    # SAVE THE CHECKPOINT WITH THE LOWEST VALIDATION LOSS
    if epoch_val_loss < best_finetune_val_loss:

        best_finetune_val_loss = epoch_val_loss
        best_finetune_epoch = epoch + 1

        torch.save(
    hqnn_model.state_dict(),
    "/content/BEST_HQNN.pth"
)

        checkpoint_message = " <-- BEST CHECKPOINT"

    else:

        checkpoint_message = ""

    # PRINT THE EPOCH RESULTS
    print(
        f"Epoch {epoch + 1:02d}: "
        f"Train Loss {epoch_train_loss:.4f}, "
        f"Acc {epoch_train_accuracy:.4f}, "
        f"F1 {epoch_train_f1:.4f} | "
        f"Val Loss {epoch_val_loss:.4f}, "
        f"Acc {epoch_val_accuracy:.4f}, "
        f"F1 {epoch_val_f1:.4f}"
        f"{checkpoint_message}"
    )

    # UPDATE EARLY STOPPING AFTER THE FIRST FIVE EPOCHS
    if epoch + 1 >= EARLY_STOPPING_START_EPOCH:

        if (
            epoch_val_loss
            < early_stopping_best_loss - MIN_DELTA
        ):

            early_stopping_best_loss = epoch_val_loss
            patience_counter = 0

        else:

            patience_counter += 1

            print(
                f"Early stopping counter: "
                f"{patience_counter}/{PATIENCE}"
            )

            if patience_counter >= PATIENCE:

                print(
                    f"Early stopping triggered "
                    f"at epoch {epoch + 1}."
                )

                break

    else:

        if epoch_val_loss < early_stopping_best_loss:

            early_stopping_best_loss = epoch_val_loss


# CALCULATE TOTAL FINE-TUNING WALL TIME
finetune_end_time = time.time()

hqnn_finetune_time = (
    finetune_end_time
    - finetune_start_time
)

finetune_minutes = int(
    hqnn_finetune_time // 60
)

finetune_seconds = int(
    hqnn_finetune_time % 60
)

# CALCULATE THE COMPLETE HQNN TRAINING TIME
total_hqnn_training_time = (
    hqnn_training_time
    + hqnn_finetune_time
)

total_minutes = int(
    total_hqnn_training_time // 60
)

total_seconds = int(
    total_hqnn_training_time % 60
)

print("\nNOISE-FREE HQNN FINE-TUNING COMPLETED")

print(
    "Best fine-tuning checkpoint epoch:",
    best_finetune_epoch
)

print(
    f"Best fine-tuning validation loss: "
    f"{best_finetune_val_loss:.4f}"
)

print(
    f"Fine-tuning wall time: "
    f"{finetune_minutes} min "
    f"{finetune_seconds} s"
)

print(
    f"Total HQNN training wall time: "
    f"{total_minutes} min "
    f"{total_seconds} s"
)

Epoch 01: Train Loss 1.2642, Acc 0.4403, F1 0.4287 | Val Loss 0.9118, Acc 0.6343, F1 0.6369 <-- BEST CHECKPOINT
Epoch 02: Train Loss 0.8562, Acc 0.6734, F1 0.6711 | Val Loss 0.6775, Acc 0.7667, F1 0.7675 <-- BEST CHECKPOINT
Epoch 03: Train Loss 0.6532, Acc 0.7766, F1 0.7762 | Val Loss 0.5553, Acc 0.8157, F1 0.8150 <-- BEST CHECKPOINT
Epoch 04: Train Loss 0.5428, Acc 0.8214, F1 0.8203 | Val Loss 0.5036, Acc 0.8463, F1 0.8446 <-- BEST CHECKPOINT
Epoch 05: Train Loss 0.4862, Acc 0.8444, F1 0.8433 | Val Loss 0.4442, Acc 0.8509, F1 0.8490 <-- BEST CHECKPOINT
Epoch 06: Train Loss 0.4432, Acc 0.8627, F1 0.8615 | Val Loss 0.4079, Acc 0.8741, F1 0.8726 <-- BEST CHECKPOINT
Epoch 07: Train Loss 0.3997, Acc 0.8754, F1 0.8743 | Val Loss 0.3966, Acc 0.8667, F1 0.8645 <-- BEST CHECKPOINT
Epoch 08: Train Loss 0.3633, Acc 0.8873, F1 0.8866 | Val Loss 0.3624, Acc 0.8852, F1 0.8837 <-- BEST CHECKPOINT
Epoch 09: Train Loss 0.3543, Acc 0.8923, F1 0.8916 | Val Loss 0.3518, Acc 0.8843, F1 0.8831 <-- BEST CHE

In [16]:
# DOWNLOAD FINAL BEST HQNN TO COMPUTER

from google.colab import files

print("Downloading BEST_HQNN.pth...")

files.download("/content/BEST_HQNN.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
# FINAL TEST OF THE BEST FINE-TUNED NOISE-FREE HQNN

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# LOAD THE BEST FINE-TUNED HQNN CHECKPOINT

best_hqnn_state = torch.load(
    "/content/BEST_HQNN.pth",
    map_location="cpu",
    weights_only=True
)

hqnn_model.load_state_dict(
    best_hqnn_state
)

print("Best fine-tuned HQNN checkpoint loaded.")


# RESTORE THE MIXED CPU-GPU DEVICE CONFIGURATION

hqnn_model.features = (
    hqnn_model.features.to(finetune_device)
)

hqnn_model.avgpool = (
    hqnn_model.avgpool.to(finetune_device)
)

hqnn_model.feature_reduction = (
    hqnn_model.feature_reduction.to(finetune_device)
)

hqnn_model.classifier = (
    hqnn_model.classifier.to(finetune_device)
)

hqnn_model.quantum_layer = (
    hqnn_model.quantum_layer.cpu()
)


# SET THE COMPLETE MODEL TO EVALUATION MODE

hqnn_model.eval()


# STORE FINAL TEST OUTPUTS

all_test_targets = []
all_test_predictions = []
all_test_probabilities = []


# EVALUATE THE MODEL ON THE COMPLETE TEST DATASET

with torch.no_grad():

    for images, labels in test_loader:

        # MOVE MRI IMAGES AND LABELS TO THE CLASSICAL DEVICE

        images = images.to(
            finetune_device
        )

        labels = labels.to(
            finetune_device
        )

        # RUN THE COMPLETE FINE-TUNED HQNN

        outputs = hqnn_model(
            images
        )

        # CONVERT LOGITS TO CLASS PROBABILITIES

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        # SELECT THE CLASS WITH THE HIGHEST LOGIT

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        # STORE TARGET LABELS

        all_test_targets.extend(
            labels.cpu().numpy()
        )

        # STORE PREDICTED LABELS

        all_test_predictions.extend(
            predictions.cpu().numpy()
        )

        # STORE CLASS PROBABILITIES

        all_test_probabilities.extend(
            probabilities.cpu().numpy()
        )


# CONVERT RESULTS TO NUMPY ARRAYS

all_test_targets = np.array(
    all_test_targets
)

all_test_predictions = np.array(
    all_test_predictions
)

all_test_probabilities = np.array(
    all_test_probabilities
)


# CALCULATE FINAL TEST METRICS

test_accuracy = accuracy_score(
    all_test_targets,
    all_test_predictions
)

test_precision = precision_score(
    all_test_targets,
    all_test_predictions,
    average="macro"
)

test_recall = recall_score(
    all_test_targets,
    all_test_predictions,
    average="macro"
)

test_f1 = f1_score(
    all_test_targets,
    all_test_predictions,
    average="macro"
)

test_roc_auc = roc_auc_score(
    all_test_targets,
    all_test_probabilities,
    multi_class="ovr",
    average="macro"
)


# PRINT FINAL TEST RESULTS

print(
    "\nFINAL FINE-TUNED NOISE-FREE HQNN TEST RESULTS"
)

print(
    f"Accuracy:        {test_accuracy:.4f}"
)

print(
    f"Macro Precision: {test_precision:.4f}"
)

print(
    f"Macro Recall:    {test_recall:.4f}"
)

print(
    f"Macro F1-score:  {test_f1:.4f}"
)

print(
    f"Macro ROC-AUC:   {test_roc_auc:.4f}"
)


# PRINT CLASSIFICATION RESULTS FOR EACH TUMOR CLASS

print(
    "\nCLASSIFICATION REPORT"
)

print(
    classification_report(
        all_test_targets,
        all_test_predictions,
        target_names=[
            "Glioma",
            "Meningioma",
            "No Tumor",
            "Pituitary"
        ],
        digits=4
    )
)

Best fine-tuned HQNN checkpoint loaded.

FINAL FINE-TUNED NOISE-FREE HQNN TEST RESULTS
Accuracy:        0.8889
Macro Precision: 0.8921
Macro Recall:    0.8889
Macro F1-score:  0.8873
Macro ROC-AUC:   0.9818

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      Glioma     0.9543    0.7741    0.8548       270
  Meningioma     0.8255    0.8407    0.8330       270
    No Tumor     0.8923    0.9815    0.9347       270
   Pituitary     0.8962    0.9593    0.9267       270

    accuracy                         0.8889      1080
   macro avg     0.8921    0.8889    0.8873      1080
weighted avg     0.8921    0.8889    0.8873      1080

